# LeetCode 1534 — Count Good Triplets bằng Python

**Link đề:** https://leetcode.com/problems/count-good-triplets/
**Độ khó:** Easy · **Chủ đề:** Array, Enumeration

```python
class Solution:
    def countGoodTriplets(self, arr: List[int], a: int, b: int, c: int) -> int:
        ...
```

### Mục tiêu học
1. Đọc đúng điều kiện **bộ ba chỉ số** `i < j < k` và ba bất đẳng thức trị tuyệt đối.
2. Hiểu `abs()`, và cách viết `|x - y| <= d` thành **khoảng** `y - d <= x <= y + d`.
3. Viết **3 vòng lặp lồng nhau** đúng giới hạn (`range(i+1, n)`, `range(j+1, n)`), đếm số lần lặp.
4. Tối ưu từng bước: brute force `O(n³)` → **cắt tỉa sớm** → **đếm bằng mảng tần suất + tổng tiền tố** `O(n² + n·V)`.
5. Trace, chứng minh, phân tích độ phức tạp, test đối chiếu, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — đếm bộ ba "tốt"

Cho mảng số nguyên `arr` và ba số nguyên `a`, `b`, `c`. Bộ ba `(arr[i], arr[j], arr[k])` là **tốt** nếu:

- `0 <= i < j < k < arr.length`
- `|arr[i] - arr[j]| <= a`
- `|arr[j] - arr[k]| <= b`
- `|arr[i] - arr[k]| <= c`

Trả về **số** bộ ba tốt.

| Ví dụ | `arr` | `a, b, c` | Output |
|---|---|---|---|
| 1 | `[3,0,1,1,9,7]` | `7, 2, 3` | `4` — các bộ: `(3,0,1), (3,0,1), (3,1,1), (0,1,1)` |
| 2 | `[1,1,2,2,3]` | `0, 0, 1` | `0` |

**Constraints:** `3 <= arr.length <= 100`, `0 <= arr[i] <= 1000`, `0 <= a, b, c <= 1000`.

Lưu ý quan trọng:
- Ta đếm theo **bộ chỉ số** `(i, j, k)`, không phải theo bộ giá trị. Ví dụ 1 có `(3,0,1)` xuất hiện **hai lần** vì có hai số `1` ở vị trí 2 và 3.
- `i < j < k` giữ **thứ tự** trong mảng — không được sắp xếp mảng rồi đếm tùy tiện.
- Mỗi cặp có **ngưỡng riêng**: cặp `(i,j)` dùng `a`, cặp `(j,k)` dùng `b`, cặp `(i,k)` dùng `c`.

In [1]:
arr = [3, 0, 1, 1, 9, 7]
for i, v in enumerate(arr):
    print(f"arr[{i}] = {v}")
print("Có", arr.count(1), "số 1 → (3,0,1) được đếm 2 lần theo chỉ số (0,1,2) và (0,1,3)")

arr[0] = 3
arr[1] = 0
arr[2] = 1
arr[3] = 1
arr[4] = 9
arr[5] = 7
Có 2 số 1 → (3,0,1) được đếm 2 lần theo chỉ số (0,1,2) và (0,1,3)


## 2. Nền tảng Python cần dùng

### 2.1. `abs()` — trị tuyệt đối

`abs(x)` trả về khoảng cách từ `x` tới 0. `abs(x - y)` là **khoảng cách** giữa hai số trên trục số, không quan tâm số nào lớn hơn.

In [2]:
print(abs(5), abs(-5), abs(0))
print("|3 - 0| =", abs(3 - 0), "| |0 - 3| =", abs(0 - 3))
print("|1 - 9| <= 3 ?", abs(1 - 9) <= 3)

5 5 0
|3 - 0| = 3 | |0 - 3| = 3
|1 - 9| <= 3 ? False


### 2.2. `|x - y| <= d` ⇔ `y - d <= x <= y + d`

"Khoảng cách từ `x` tới `y` không quá `d`" nghĩa là `x` nằm trong **đoạn** `[y-d, y+d]`. Python cho phép viết **so sánh chuỗi** `lo <= x <= hi`. Phép biến đổi này là chìa khóa của cách tối ưu.

In [3]:
y, d = 5, 2
for x in range(1, 10):
    cond1 = abs(x - y) <= d
    cond2 = y - d <= x <= y + d
    print(x, cond1, cond2)
    assert cond1 == cond2

1 False False
2 False False
3 True True
4 True True
5 True True
6 True True
7 True True
8 False False
9 False False


### 2.3. Ba vòng lặp lồng nhau với `i < j < k`

Để sinh mọi bộ chỉ số tăng dần, vòng trong **bắt đầu sau** vòng ngoài: `j` chạy từ `i+1`, `k` chạy từ `j+1`. Như vậy mỗi bộ được sinh **đúng một lần**, không trùng, không hoán vị.

In [4]:
n = 4
triples = []
for i in range(n):
    for j in range(i + 1, n):
        for k in range(j + 1, n):
            triples.append((i, j, k))
print(triples)
print("Số bộ =", len(triples), "= C(4,3) =", 4 * 3 * 2 // 6)

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]
Số bộ = 4 = C(4,3) = 4


Số bộ ba là tổ hợp `C(n, 3) = n(n-1)(n-2)/6`. Với `n = 100`: `161 700` bộ — rất nhỏ với máy tính, nên brute force **chắc chắn qua** (đề cũng gợi ý như vậy). Thư viện chuẩn có `math.comb` để tính nhanh:

In [5]:
import math
for n in [3, 10, 100, 1000]:
    print(f"n={n:>4}: C(n,3) = {math.comb(n, 3):,}")

n=   3: C(n,3) = 1
n=  10: C(n,3) = 120
n= 100: C(n,3) = 161,700
n=1000: C(n,3) = 166,167,000


`f"{x:,}"` in số có dấu phẩy ngăn cách hàng nghìn cho dễ đọc. Với `n = 1000` đã là 166 triệu — đó là lúc cần cách tốt hơn.

### 2.4. `itertools.combinations`

`combinations(iterable, 3)` sinh mọi bộ 3 phần tử **theo thứ tự xuất hiện** — tương đương 3 vòng lặp ở trên, viết gọn hơn.

In [6]:
from itertools import combinations
print(list(combinations([3, 0, 1, 1], 3)))
print(list(combinations(range(4), 3)))

[(3, 0, 1), (3, 0, 1), (3, 1, 1), (0, 1, 1)]
[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]


## 3. Cách 1 — Brute force: thử mọi bộ ba

Ý tưởng tự nhiên nhất: duyệt mọi `(i, j, k)` với `i < j < k`, kiểm tra cả 3 điều kiện, đúng thì `count += 1`.

In [7]:
def count_brute(arr, a, b, c):
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if (abs(arr[i] - arr[j]) <= a and
                        abs(arr[j] - arr[k]) <= b and
                        abs(arr[i] - arr[k]) <= c):
                    count += 1
    return count

print(count_brute([3, 0, 1, 1, 9, 7], 7, 2, 3))   # 4
print(count_brute([1, 1, 2, 2, 3], 0, 0, 1))      # 0

4
0


Ngoặc tròn bao quanh điều kiện dài cho phép **xuống dòng** giữa biểu thức mà không cần dấu `\`. Toán tử `and` **ngắn mạch** (short-circuit): nếu vế đầu `False`, Python không tính các vế sau.

Phiên bản dùng `combinations` — gọn, cùng độ phức tạp:

In [8]:
def count_brute_combo(arr, a, b, c):
    return sum(1 for x, y, z in combinations(arr, 3)
               if abs(x - y) <= a and abs(y - z) <= b and abs(x - z) <= c)

print(count_brute_combo([3, 0, 1, 1, 9, 7], 7, 2, 3))

4


`sum(1 for ... if ...)` đếm số phần tử thỏa điều kiện: mỗi phần tử đạt cộng 1. `True/False` trong Python cũng là `1/0`, nên `sum(cond for ...)` cũng đếm được.

## 4. Nút thắt — vòng `k` chạy cả khi `(i, j)` đã hỏng

Quan sát: điều kiện `|arr[i] - arr[j]| <= a` **không phụ thuộc `k`**. Nếu cặp `(i, j)` đã vi phạm thì mọi `k` đều vô ích — nhưng brute force vẫn duyệt hết `k`.

Đếm số lần chạy vòng trong nhất để thấy lãng phí:

In [9]:
def count_inner_steps(arr, a, prune):
    n = len(arr); steps = 0
    for i in range(n):
        for j in range(i + 1, n):
            if prune and abs(arr[i] - arr[j]) > a:
                continue
            for k in range(j + 1, n):
                steps += 1
    return steps

import random
rng = random.Random(1534)
arr = [rng.randint(0, 1000) for _ in range(100)]
print("Không cắt tỉa:", count_inner_steps(arr, 50, prune=False))
print("Có cắt tỉa   :", count_inner_steps(arr, 50, prune=True))

Không cắt tỉa: 161700
Có cắt tỉa   : 17232


## 5. Cách 2 — Brute force có cắt tỉa (pruning)

Kiểm tra điều kiện `a` **ngay sau khi có `i, j`**; hỏng thì `continue` (bỏ qua phần còn lại của vòng `j` hiện tại, sang `j` kế tiếp).

### 5.1. Từng đoạn
1. Vòng `i`, vòng `j` như cũ.
2. `if abs(arr[i] - arr[j]) > a: continue` — cắt cả nhánh `k`.
3. Vòng `k`: chỉ còn kiểm tra điều kiện `b` và `c`.

### 5.2. Ghép lại

In [10]:
def count_pruned(arr, a, b, c):
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(arr[i] - arr[j]) > a:
                continue                      # cặp (i,j) hỏng → bỏ mọi k
            for k in range(j + 1, n):
                if abs(arr[j] - arr[k]) <= b and abs(arr[i] - arr[k]) <= c:
                    count += 1
    return count

print(count_pruned([3, 0, 1, 1, 9, 7], 7, 2, 3))
print(count_pruned([1, 1, 2, 2, 3], 0, 0, 1))

4
0


Đây là **lời giải chính** khuyên dùng cho Easy: dễ đọc, đúng, nhanh hơn brute force thuần trong thực tế. Tuy vậy **worst case vẫn `O(n³)`** (khi `a` lớn, không cắt được gì).

## 6. TRACE từng bước — ví dụ 1

`arr = [3,0,1,1,9,7]`, `a=7, b=2, c=3`. Bảng dưới liệt kê các cặp `(i,j)` **qua** điều kiện `a` và các `k` được đếm:

| (i,j) | giá trị | qua `a`? | k được đếm |
|---|---|---|---|
| (0,1) | 3,0 | 3 ≤ 7 ✓ | k=2 (1), k=3 (1) |
| (0,2) | 3,1 | 2 ≤ 7 ✓ | k=3 (1) |
| (0,4) | 3,9 | 6 ≤ 7 ✓ | không (|9-7|=2 ≤ b nhưng |3-7|=4 > c) |
| (1,2) | 0,1 | ✓ | k=3 (1) |
| (1,4) | 0,9 | 9 > 7 ✗ | — cắt |

Tổng: 2 + 1 + 1 = **4**.

In [11]:
def trace_pruned(arr, a, b, c):
    n = len(arr); count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(arr[i] - arr[j]) > a:
                print(f"(i={i},j={j}) vals=({arr[i]},{arr[j]}) |diff|={abs(arr[i]-arr[j])} > a → CẮT")
                continue
            for k in range(j + 1, n):
                okb = abs(arr[j] - arr[k]) <= b
                okc = abs(arr[i] - arr[k]) <= c
                if okb and okc:
                    count += 1
                    print(f"  ĐẾM (i={i},j={j},k={k}) = ({arr[i]},{arr[j]},{arr[k]}) → count={count}")
    print("Kết quả:", count)
    return count

assert trace_pruned([3, 0, 1, 1, 9, 7], 7, 2, 3) == 4

  ĐẾM (i=0,j=1,k=2) = (3,0,1) → count=1
  ĐẾM (i=0,j=1,k=3) = (3,0,1) → count=2
  ĐẾM (i=0,j=2,k=3) = (3,1,1) → count=3
  ĐẾM (i=1,j=2,k=3) = (0,1,1) → count=4
(i=1,j=4) vals=(0,9) |diff|=9 > a → CẮT
(i=2,j=4) vals=(1,9) |diff|=8 > a → CẮT
(i=3,j=4) vals=(1,9) |diff|=8 > a → CẮT
Kết quả: 4


Trace chỉ in dòng **CẮT** và dòng **ĐẾM** để không quá dài. Chú ý các cặp có giá trị 9 với 0 (chênh 9 > 7) bị cắt ngay — đó là chỗ tiết kiệm vòng `k`.

## 7. Cách 3 — Cố định `j, k`, đếm `i` bằng tổng tiền tố

### 7.1. Insight
Cố định cặp `(j, k)` với `j < k` và `|arr[j] - arr[k]| <= b`. Còn lại ta cần đếm số `i < j` sao cho:

- `arr[j] - a <= arr[i] <= arr[j] + a`
- `arr[k] - c <= arr[i] <= arr[k] + c`

Giao của hai đoạn là **một đoạn** `[lo, hi]` với `lo = max(arr[j]-a, arr[k]-c)`, `hi = min(arr[j]+a, arr[k]+c)`. Bài toán con: **có bao nhiêu `arr[i]` (với `i < j`) rơi vào `[lo, hi]`?**

### 7.2. Mảng tần suất + tổng tiền tố
Giá trị nằm trong `[0, 1000]`, nên ta dùng mảng `prefix` độ dài `V + 1` (V = 1001): `prefix[v]` = số phần tử đã gặp (chỉ số `< j`) có giá trị `<= v`. Khi đó:

```text
số arr[i] trong [lo, hi] = prefix[hi] - prefix[lo - 1]     (nếu lo > 0)
```

Mỗi khi `j` tăng thêm 1, phần tử `arr[j]` được "thêm vào quá khứ": mọi `prefix[v]` với `v >= arr[j]` tăng 1 → tốn `O(V)`.

Ví dụ nhỏ về tổng tiền tố:

In [12]:
vals_before = [3, 0, 1]          # các arr[i] đã gặp
V = 10
freq = [0] * V
for v in vals_before:
    freq[v] += 1
prefix = [0] * V
running = 0
for v in range(V):
    running += freq[v]
    prefix[v] = running
print("freq  :", freq)
print("prefix:", prefix)
lo, hi = 1, 3
cnt = prefix[hi] - (prefix[lo - 1] if lo > 0 else 0)
print(f"Số giá trị trong [{lo},{hi}] =", cnt)   # 1 và 3 → 2

freq  : [1, 1, 0, 1, 0, 0, 0, 0, 0, 0]
prefix: [1, 2, 2, 3, 3, 3, 3, 3, 3, 3]
Số giá trị trong [1,3] = 2


### 7.3. Ghép lại

In [13]:
def count_prefix(arr, a, b, c):
    V = 1001                               # arr[i] ∈ [0, 1000]
    prefix = [0] * V                       # prefix[v] = #arr[i] (i < j) có giá trị <= v
    n = len(arr)
    count = 0
    for j in range(n):
        for k in range(j + 1, n):
            if abs(arr[j] - arr[k]) > b:
                continue
            lo = max(arr[j] - a, arr[k] - c, 0)
            hi = min(arr[j] + a, arr[k] + c, V - 1)
            if lo > hi:
                continue
            count += prefix[hi] - (prefix[lo - 1] if lo > 0 else 0)
        for v in range(arr[j], V):         # đưa arr[j] vào "quá khứ" cho các j sau
            prefix[v] += 1
    return count

print(count_prefix([3, 0, 1, 1, 9, 7], 7, 2, 3))
print(count_prefix([1, 1, 2, 2, 3], 0, 0, 1))

4
0


Điểm mấu chốt về **thứ tự**: ta đếm các cặp `(j, k)` **trước**, rồi mới thêm `arr[j]` vào `prefix`. Nhờ vậy khi xét `j`, `prefix` chỉ chứa `arr[0..j-1]` — đúng điều kiện `i < j`.

`lo` được kẹp bằng `max(..., 0)` và `hi` kẹp bằng `min(..., V-1)` để không truy cập ngoài mảng.

### 7.4. Trace cách 3

In [14]:
def trace_prefix(arr, a, b, c):
    V = 1001; prefix = [0] * V; n = len(arr); count = 0
    for j in range(n):
        print(f"j={j}, arr[j]={arr[j]}, quá khứ = {arr[:j]}")
        for k in range(j + 1, n):
            if abs(arr[j] - arr[k]) > b:
                continue
            lo = max(arr[j] - a, arr[k] - c, 0); hi = min(arr[j] + a, arr[k] + c, V - 1)
            add = prefix[hi] - (prefix[lo - 1] if lo > 0 else 0) if lo <= hi else 0
            count += add
            print(f"   k={k} arr[k]={arr[k]} → khoảng [{lo},{hi}], cộng {add}, count={count}")
        for v in range(arr[j], V):
            prefix[v] += 1
    return count

assert trace_prefix([3, 0, 1, 1, 9, 7], 7, 2, 3) == 4

j=0, arr[j]=3, quá khứ = []
   k=2 arr[k]=1 → khoảng [0,4], cộng 0, count=0
   k=3 arr[k]=1 → khoảng [0,4], cộng 0, count=0
j=1, arr[j]=0, quá khứ = [3]
   k=2 arr[k]=1 → khoảng [0,4], cộng 1, count=1
   k=3 arr[k]=1 → khoảng [0,4], cộng 1, count=2
j=2, arr[j]=1, quá khứ = [3, 0]
   k=3 arr[k]=1 → khoảng [0,4], cộng 2, count=4
j=3, arr[j]=1, quá khứ = [3, 0, 1]
j=4, arr[j]=9, quá khứ = [3, 0, 1, 1]
   k=5 arr[k]=7 → khoảng [4,10], cộng 0, count=4
j=5, arr[j]=7, quá khứ = [3, 0, 1, 1, 9]


Đọc trace: với `j=2` (giá trị 1), `k=3` (giá trị 1): khoảng `[max(1-7,1-3,0), min(1+7,1+3)] = [0, 4]`, quá khứ `[3, 0]` có 2 giá trị trong đó → cộng 2. Với `j=1, k=2` và `j=1, k=3`: quá khứ `[3]` thuộc `[0, 4]` → mỗi cặp cộng 1. Tổng 4. ✓

## 8. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi `(i,j,k)` | `O(n³)` | `O(1)` | `n` nhỏ (≤ ~200), viết nhanh |
| 2. Cắt tỉa | bỏ vòng `k` khi `(i,j)` hỏng | `O(n³)` worst, nhanh hơn thực tế | `O(1)` | **lời giải nộp khuyên dùng** |
| 3. Tiền tố | cố định `(j,k)`, đếm `i` bằng prefix | `O(n² + n·V)` | `O(V)` | `n` lớn, giá trị nhỏ |

Với `n = 100`, `V = 1001`: cách 3 tốn ~`10⁴ + 10⁵` phép, cách 1 tốn ~`1.6·10⁵` — tương đương nhau! Cách 3 chỉ thật sự thắng khi `n` lớn (ví dụ `n = 3000`).

## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def countGoodTriplets(self, arr: List[int], a: int, b: int, c: int) -> int:
        n = len(arr)
        count = 0
        for i in range(n):
            for j in range(i + 1, n):
                if abs(arr[i] - arr[j]) > a:
                    continue
                for k in range(j + 1, n):
                    if abs(arr[j] - arr[k]) <= b and abs(arr[i] - arr[k]) <= c:
                        count += 1
        return count

sol = Solution()
print(sol.countGoodTriplets([3, 0, 1, 1, 9, 7], 7, 2, 3))
print(sol.countGoodTriplets([1, 1, 2, 2, 3], 0, 0, 1))

4
0


`from typing import List` cho phép viết gợi ý kiểu `List[int]` như skeleton của LeetCode (trên LeetCode đã import sẵn). Từ Python 3.9 có thể viết `list[int]`.

Phiên bản nộp dùng cách 3 (tham khảo):

```python
class Solution:
    def countGoodTriplets(self, arr: List[int], a: int, b: int, c: int) -> int:
        V = 1001
        prefix = [0] * V
        count = 0
        for j in range(len(arr)):
            for k in range(j + 1, len(arr)):
                if abs(arr[j] - arr[k]) <= b:
                    lo = max(arr[j] - a, arr[k] - c, 0)
                    hi = min(arr[j] + a, arr[k] + c, V - 1)
                    if lo <= hi:
                        count += prefix[hi] - (prefix[lo - 1] if lo > 0 else 0)
            for v in range(arr[j], V):
                prefix[v] += 1
        return count
```

## 10. Vì sao đúng?

**Cách 1:** ba vòng lặp với `j > i`, `k > j` sinh **mỗi bộ chỉ số tăng dần đúng một lần** (song ánh với các tập con 3 phần tử). Ta đếm bộ khi và chỉ khi cả ba điều kiện đúng → kết quả chính xác.

**Cách 2:** chỉ bỏ những bộ `(i,j,k)` có `|arr[i]-arr[j]| > a` — những bộ này **chắc chắn không tốt** → không bỏ sót bộ tốt nào.

**Cách 3 — invariant:** ở đầu vòng lặp ngoài với chỉ số `j`, `prefix[v]` = số chỉ số `i < j` có `arr[i] <= v`.
- Khởi tạo `j = 0`: không có `i` nào, `prefix` toàn 0 ✓.
- Cuối vòng `j`, cộng 1 cho mọi `v >= arr[j]` → đầu vòng `j+1` invariant đúng ✓.
- Với cặp `(j,k)` thỏa `b`, các `i` hợp lệ là `i < j` có `arr[i]` thuộc giao hai đoạn `[lo, hi]`; `prefix[hi] - prefix[lo-1]` đếm đúng số đó. Kẹp `lo ≥ 0`, `hi ≤ 1000` không đổi kết quả vì mọi `arr[i]` đều trong `[0, 1000]`.

## 11. Time / Space

Gọi `n = len(arr)`, `V = 1001` (số giá trị có thể).

| Cách | Time | Space |
|---|---|---|
| Brute force | `C(n,3) ≈ n³/6` → `O(n³)` | `O(1)` |
| Cắt tỉa | worst `O(n³)` (khi `a ≥ 1000`) | `O(1)` |
| Tiền tố | cặp `(j,k)`: `O(n²)`; cập nhật prefix: `n` lần × `O(V)` → `O(n² + n·V)` | `O(V)` |

Với giới hạn đề (`n ≤ 100`) mọi cách đều nhanh. Cải tiến thêm: dùng **Fenwick tree** để cập nhật/truy vấn `O(log V)` → `O(n² log V)`; không cần cho bài này.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.countGoodTriplets([3, 0, 1, 1, 9, 7], 7, 2, 3), 4)
check("đề 2", sol.countGoodTriplets([1, 1, 2, 2, 3], 0, 0, 1), 0)
check("n=3 tốt", sol.countGoodTriplets([1, 2, 3], 1, 1, 2), 1)
check("n=3 hỏng c", sol.countGoodTriplets([1, 2, 3], 1, 1, 1), 0)
check("toàn bằng nhau, ngưỡng 0", sol.countGoodTriplets([5, 5, 5, 5], 0, 0, 0), 4)   # C(4,3)
check("ngưỡng max", sol.countGoodTriplets([0, 1000, 0, 1000], 1000, 1000, 1000), 4)
check("giá trị biên 0/1000", sol.countGoodTriplets([0, 1000, 0], 999, 1000, 0), 0)

PASS đề 1: 4
PASS đề 2: 0
PASS n=3 tốt: 1
PASS n=3 hỏng c: 0
PASS toàn bằng nhau, ngưỡng 0: 4
PASS ngưỡng max: 4
PASS giá trị biên 0/1000: 0


**Phản ví dụ "sắp xếp trước":** người mới có thể sắp xếp `arr` cho dễ xử lý. Nhưng điều kiện gắn ngưỡng **khác nhau** cho từng cặp vị trí, nên đổi thứ tự làm sai:

In [17]:
arr, a, b, c = [9, 0, 7], 9, 7, 2
print("Đúng (giữ thứ tự):", count_brute(arr, a, b, c))
print("Sai (sắp xếp trước):", count_brute(sorted(arr), a, b, c))

Đúng (giữ thứ tự): 1
Sai (sắp xếp trước): 0


Với `[9,0,7]`: `|9-0|=9≤a`, `|0-7|=7≤b`, `|9-7|=2≤c` → tốt (1). Sau khi sắp thành `[0,7,9]`: `|0-9|=9 > c=2` → 0. Kết quả khác nhau → **không được sắp xếp**.

### 12.1. Đối chiếu 3 cách trên input ngẫu nhiên

In [18]:
rng = random.Random(2024)
for _ in range(300):
    n = rng.randint(3, 25)
    arr = [rng.randint(0, rng.choice([5, 50, 1000])) for _ in range(n)]
    a, b, c = (rng.randint(0, 1000 if rng.random() < 0.3 else 20) for _ in range(3))
    exp = count_brute(arr, a, b, c)
    assert count_brute_combo(arr, a, b, c) == exp
    assert count_pruned(arr, a, b, c) == exp
    assert count_prefix(arr, a, b, c) == exp
    assert Solution().countGoodTriplets(arr, a, b, c) == exp
print("300 test ngẫu nhiên: 3 cách khớp nhau")

300 test ngẫu nhiên: 3 cách khớp nhau


### 12.2. Đo thời gian khi `n` lớn hơn đề

`time.perf_counter()` cho thời điểm hiện tại với độ chính xác cao; lấy hiệu hai lần gọi để đo thời gian chạy. (Con số phụ thuộc máy, chỉ để so sánh tương đối.)

In [19]:
import time
rng = random.Random(1)
arr = [rng.randint(0, 1000) for _ in range(300)]
for name, f in [("brute", count_brute), ("pruned", count_pruned), ("prefix", count_prefix)]:
    t0 = time.perf_counter()
    r = f(arr, 100, 100, 100)
    print(f"{name:>6}: kết quả={r}, thời gian ~ {time.perf_counter() - t0:.3f}s")

 brute: kết quả=119512, thời gian ~ 0.242s
pruned: kết quả=119512, thời gian ~ 0.058s
prefix: kết quả=119512, thời gian ~ 0.009s


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Vòng trong bắt đầu từ 0
```python
for i in range(n):
    for j in range(n):
        for k in range(n):
```
Đếm cả bộ `i == j`, bộ lặp lại theo hoán vị → kết quả quá lớn. **Sửa:** `range(i + 1, n)`, `range(j + 1, n)`.

In [20]:
def count_wrong_ranges(arr, a, b, c):
    n = len(arr); cnt = 0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                if abs(arr[i]-arr[j]) <= a and abs(arr[j]-arr[k]) <= b and abs(arr[i]-arr[k]) <= c:
                    cnt += 1
    return cnt

print("sai:", count_wrong_ranges([3, 0, 1, 1, 9, 7], 7, 2, 3), "| đúng:", count_brute([3, 0, 1, 1, 9, 7], 7, 2, 3))

sai: 64 | đúng: 4


### Lỗi 2 — Quên `abs`
`arr[i] - arr[j] <= a` chỉ chặn một phía: khi `arr[i] < arr[j]` hiệu âm luôn `<= a` → đếm thừa.

In [21]:
def count_no_abs(arr, a, b, c):
    n = len(arr); cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if arr[i]-arr[j] <= a and arr[j]-arr[k] <= b and arr[i]-arr[k] <= c:
                    cnt += 1
    return cnt

print("quên abs:", count_no_abs([0, 5, 10], 1, 1, 1), "| đúng:", count_brute([0, 5, 10], 1, 1, 1))

quên abs: 1 | đúng: 0


### Lỗi 3 — Gán nhầm ngưỡng
Dùng `a` cho cặp `(j,k)` hoặc `c` cho `(i,j)`. Đề ghép: `a ↔ (i,j)`, `b ↔ (j,k)`, `c ↔ (i,k)`. Mẹo nhớ: thứ tự `a, b, c` theo các cặp (đầu-giữa), (giữa-cuối), (đầu-cuối).

### Lỗi 4 — `break` thay vì `continue` khi cắt tỉa
`break` ở vòng `j` thoát **toàn bộ** vòng `j`, bỏ luôn các `j` sau có thể vẫn tốt.

In [22]:
def count_break_bug(arr, a, b, c):
    n = len(arr); cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(arr[i] - arr[j]) > a:
                break                       # SAI
            for k in range(j + 1, n):
                if abs(arr[j]-arr[k]) <= b and abs(arr[i]-arr[k]) <= c:
                    cnt += 1
    return cnt

arr = [1, 5, 1, 1]
print("break:", count_break_bug(arr, 1, 1, 1), "| continue:", count_pruned(arr, 1, 1, 1))
print("Bộ (0,2,3) = (1,1,1) bị bỏ sót vì j=1 (giá trị 5) làm break cả vòng j")

break: 0 | continue: 1
Bộ (0,2,3) = (1,1,1) bị bỏ sót vì j=1 (giá trị 5) làm break cả vòng j


### Lỗi 5 (cách 3) — Thêm `arr[j]` vào prefix **trước** khi đếm
Khi đó `i` có thể bằng `j` → đếm cả bộ dùng chung chỉ số. Phải đếm xong các `k` rồi mới cập nhật.

## 14. Mẹo nhận dạng pattern

- "Đếm bộ ba/cặp chỉ số thỏa điều kiện" + `n ≤ 100` → **brute force** lồng vòng là đủ; đọc constraints trước khi tối ưu.
- Điều kiện chỉ phụ thuộc **một phần** biến lặp → kiểm tra **ở vòng ngoài nhất có thể** (cắt tỉa).
- `|x - y| <= d` → đổi thành **khoảng** `[y-d, y+d]`; nhiều điều kiện cùng biến → **giao các khoảng**.
- "Đếm phần tử phía trước nằm trong `[lo, hi]`" + giá trị nhỏ → **mảng tần suất + tổng tiền tố** (hoặc Fenwick tree khi cần cập nhật nhanh).
- Bộ ba `i < j < k` → thử **cố định phần tử giữa** hoặc **cố định hai phần tử** rồi đếm phần còn lại.

## 15. Biến thể

### 15.1. Đếm cặp `i < j` với `|arr[i] - arr[j]| <= d`
Cùng kỹ thuật tiền tố, giờ chỉ một vòng.

In [23]:
def count_close_pairs(arr, d, V=1001):
    prefix = [0] * V; cnt = 0
    for x in arr:
        lo, hi = max(x - d, 0), min(x + d, V - 1)
        cnt += prefix[hi] - (prefix[lo - 1] if lo > 0 else 0)
        for v in range(x, V):
            prefix[v] += 1
    return cnt

arr = [3, 0, 1, 1, 9, 7]
brute = sum(1 for i in range(len(arr)) for j in range(i + 1, len(arr)) if abs(arr[i] - arr[j]) <= 2)
print(count_close_pairs(arr, 2), brute)

6 6


### 15.2. Liệt kê các bộ ba (không chỉ đếm)
Đổi `count += 1` thành `result.append(...)`. Không thể nhanh hơn số bộ được liệt kê.

In [24]:
def list_good_triplets(arr, a, b, c):
    return [(i, j, k) for i, j, k in combinations(range(len(arr)), 3)
            if abs(arr[i]-arr[j]) <= a and abs(arr[j]-arr[k]) <= b and abs(arr[i]-arr[k]) <= c]

print(list_good_triplets([3, 0, 1, 1, 9, 7], 7, 2, 3))

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]


### 15.3. Bài liên quan
- LeetCode 1395 *Count Number of Teams* — bộ ba tăng/giảm, cố định phần tử giữa.
- LeetCode 2179 *Count Good Triplets in an Array* (Hard) — cùng tên nhưng cần Fenwick tree.

## 16. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
`arr = [1, 2, 3, 4]`, `a = b = c = 1`. Liệt kê 4 bộ chỉ số và cho biết bộ nào tốt.

### Bài 2 — Đoán output
```python
print(sum(1 for i in range(5) for j in range(i + 1, 5) for k in range(j + 1, 5)))
```

### Bài 3 — TODO: viết bản cắt tỉa

In [25]:
def exercise_count(arr, a, b, c):
    # TODO 1: ba vòng lặp i < j < k
    # TODO 2: cắt tỉa ngay khi |arr[i]-arr[j]| > a
    # TODO 3: đếm khi thỏa b và c
    pass

# assert exercise_count([3, 0, 1, 1, 9, 7], 7, 2, 3) == 4
# assert exercise_count([1, 1, 2, 2, 3], 0, 0, 1) == 0

### Bài 4 — TODO: đếm cặp gần nhau bằng hai vòng
Đếm số cặp `i < j` có `|arr[i] - arr[j]| <= d`.

In [26]:
def exercise_close_pairs(arr, d):
    # TODO
    pass

# assert exercise_close_pairs([3, 0, 1, 1, 9, 7], 2) == 6

### Bài 5 — TODO: bộ ba tăng nghiêm ngặt
Đếm `i < j < k` với `arr[i] < arr[j] < arr[k]`, yêu cầu `O(n²)` (gợi ý: cố định `j`).

In [27]:
def exercise_increasing_triplets(arr):
    # TODO: với mỗi j, đếm số nhỏ hơn bên trái × số lớn hơn bên phải
    pass

# assert exercise_increasing_triplets([2, 5, 3, 4, 1]) == 1
# assert exercise_increasing_triplets([1, 2, 3, 4]) == 4

### Bài 6 — Mini challenge: cài cách tiền tố cho giá trị tới `10^6`
Với `V` lớn, cập nhật `O(V)` mỗi bước quá chậm. Hãy dùng **Fenwick tree** (Binary Indexed Tree) để cập nhật và truy vấn `O(log V)`.

In [28]:
def exercise_count_fenwick(arr, a, b, c):
    # TODO: Fenwick tree đếm số arr[i] (i < j) có giá trị <= v
    pass

# assert exercise_count_fenwick([3, 0, 1, 1, 9, 7], 7, 2, 3) == 4

## 17. Gợi ý 3 mức

**Bài 1:** (1) Có `C(4,3) = 4` bộ. (2) Kiểm tra từng cặp với ngưỡng 1. (3) Cặp (đầu, cuối) có hiệu ≥ 2 với mọi bộ.

**Bài 2:** (1) Đó là đếm bộ ba của 5 phần tử. (2) `C(5,3)`. (3) `5·4·3/6`.

**Bài 3:** (1) Ba vòng, `range(i+1,n)`. (2) `continue`, không `break`. (3) Xem mục 5.

**Bài 4:** (1) Hai vòng lồng `i < j`. (2) `abs(arr[i]-arr[j]) <= d`. (3) `sum(1 for i ... for j in range(i+1, n) if ...)`.

**Bài 5:** (1) Bộ ba tăng = (số bên trái nhỏ hơn `arr[j]`) × (số bên phải lớn hơn `arr[j]`). (2) Mỗi `j` duyệt trái và phải `O(n)`. (3) `left = sum(arr[i] < arr[j] for i in range(j))`.

**Bài 6:** (1) Fenwick lưu tổng tần suất; `update(v, +1)` và `query(v)` = tổng `[0..v]`. (2) Chỉ số Fenwick bắt đầu từ 1: dùng `v + 1`. (3) `i += i & -i` khi update, `i -= i & -i` khi query.

## 18. Đáp án tham khảo

**Bài 1:** Các bộ `(0,1,2)=(1,2,3)`: `|1-3|=2 > 1` ✗; `(0,1,3)`, `(0,2,3)`, `(1,2,3)` cũng có cặp đầu-cuối chênh ≥ 2 ✗ → **0** bộ tốt.

**Bài 2:** `C(5,3) = 10`.

In [29]:
def answer_count(arr, a, b, c):
    n = len(arr); cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(arr[i] - arr[j]) > a:
                continue
            for k in range(j + 1, n):
                if abs(arr[j] - arr[k]) <= b and abs(arr[i] - arr[k]) <= c:
                    cnt += 1
    return cnt

def answer_close_pairs(arr, d):
    n = len(arr)
    return sum(1 for i in range(n) for j in range(i + 1, n) if abs(arr[i] - arr[j]) <= d)

def answer_increasing_triplets(arr):
    n = len(arr); total = 0
    for j in range(n):
        left = sum(1 for i in range(j) if arr[i] < arr[j])
        right = sum(1 for k in range(j + 1, n) if arr[k] > arr[j])
        total += left * right
    return total

class Fenwick:
    def __init__(self, size):
        self.t = [0] * (size + 1)
    def update(self, v, delta):          # v là giá trị 0-based
        i = v + 1
        while i < len(self.t):
            self.t[i] += delta
            i += i & -i
    def query(self, v):                  # tổng tần suất các giá trị <= v
        if v < 0:
            return 0
        i = min(v + 1, len(self.t) - 1); s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

def answer_count_fenwick(arr, a, b, c, V=None):
    V = V or (max(arr) + 1)
    fw = Fenwick(V); n = len(arr); cnt = 0
    for j in range(n):
        for k in range(j + 1, n):
            if abs(arr[j] - arr[k]) <= b:
                lo = max(arr[j] - a, arr[k] - c, 0); hi = min(arr[j] + a, arr[k] + c, V - 1)
                if lo <= hi:
                    cnt += fw.query(hi) - fw.query(lo - 1)
        fw.update(arr[j], 1)
    return cnt

assert answer_count([3, 0, 1, 1, 9, 7], 7, 2, 3) == 4
assert answer_count([1, 1, 2, 2, 3], 0, 0, 1) == 0
assert answer_count([1, 2, 3, 4], 1, 1, 1) == 0
assert answer_close_pairs([3, 0, 1, 1, 9, 7], 2) == 6
assert answer_increasing_triplets([2, 5, 3, 4, 1]) == 1
assert answer_increasing_triplets([1, 2, 3, 4]) == 4
assert answer_count_fenwick([3, 0, 1, 1, 9, 7], 7, 2, 3) == 4
rng = random.Random(99)
for _ in range(200):
    arr = [rng.randint(0, 10**6 if rng.random() < 0.5 else 30) for _ in range(rng.randint(3, 20))]
    a, b, c = (rng.randint(0, 40) for _ in range(3))
    assert answer_count_fenwick(arr, a, b, c) == count_brute(arr, a, b, c)
print("Tất cả đáp án PASS (kể cả Fenwick với giá trị tới 10^6)")

Tất cả đáp án PASS (kể cả Fenwick với giá trị tới 10^6)


**Bài 4 kiểm tra tay:** `[3,0,1,1,9,7]`, `d=2`: các cặp gần: (3,1),(3,1),(0,1),(0,1),(1,1),(9,7) → 6 ✓.

## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countGoodTriplets(self, arr, a, b, c):
        n, cnt = len(arr), 0
        for i in range(n):
            for j in range(i + 1, n):
                if abs(arr[i] - arr[j]) > a:     # cắt tỉa sớm
                    continue
                for k in range(j + 1, n):
                    if abs(arr[j] - arr[k]) <= b and abs(arr[i] - arr[k]) <= c:
                        cnt += 1
        return cnt
```

**Câu thần chú:** *"Đọc constraints trước — n ≤ 100 thì O(n³) ổn; kiểm điều kiện ở vòng ngoài nhất có thể."*

**Checklist trước khi submit**
- [ ] `j` từ `i+1`, `k` từ `j+1` (không trùng, không hoán vị).
- [ ] Dùng `abs`, đúng cặp ngưỡng `a↔(i,j)`, `b↔(j,k)`, `c↔(i,k)`.
- [ ] Cắt tỉa bằng `continue`, không phải `break`.
- [ ] Không sắp xếp mảng.
- [ ] Đúng tên method `countGoodTriplets`; có `List` (LeetCode import sẵn).